# JSON Basics — 03: Schema Inference

JSON has no built-in schema — Spark must infer it from the data.
Understanding this process helps you avoid surprises in production.

Topics: inferSchema cost, samplingRatio, type conflicts, nested inference,
explicit schema vs inferred, primitivesAsString option.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:47:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — How Schema Inference Works



In [2]:

import json as pyjson, pathlib, random

# Generate data with type challenges
tricky = [
    {"id":1,"value":100,     "score":"9.5",  "tag":"vip",    "date":"2024-01-15"},
    {"id":2,"value":200,     "score":"8.0",  "tag":"gold",   "date":"2024-02-01"},
    {"id":3,"value":300.5,   "score":"7.5",  "tag":"silver", "date":"2024-03-10"},
    {"id":4,"value":"N/A",   "score":"6.0",  "tag":"bronze", "date":"2024-04-20"},
    {"id":5,"value":None,    "score":None,   "tag":None,     "date":None},
]
tricky_path = f"{DATA_DIR}/tricky.json"
with open(tricky_path,"w") as f:
    for r in tricky: f.write(pyjson.dumps(r)+"\n")

df_inferred = spark.read.json(tricky_path)
print("Inferred schema from tricky data:")
df_inferred.printSchema()
df_inferred.show(truncate=False)
print()
print("Observations:")
print("  value: LongType — row 3 has 300.5 but Spark saw int first → coerced to LONG")
print("  Actually: Spark sees int,int,double,string → uses STRING (widest common type)")
print("  score: STRING because it's quoted in source JSON")


Inferred schema from tricky data:
root
 |-- date: string (nullable = true)
 |-- id: long (nullable = true)
 |-- score: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- value: string (nullable = true)

+----------+---+-----+------+-----+
|date      |id |score|tag   |value|
+----------+---+-----+------+-----+
|2024-01-15|1  |9.5  |vip   |100  |
|2024-02-01|2  |8.0  |gold  |200  |
|2024-03-10|3  |7.5  |silver|300.5|
|2024-04-20|4  |6.0  |bronze|N/A  |
|NULL      |5  |NULL |NULL  |NULL |
+----------+---+-----+------+-----+


Observations:
  value: LongType — row 3 has 300.5 but Spark saw int first → coerced to LONG
  Actually: Spark sees int,int,double,string → uses STRING (widest common type)
  score: STRING because it's quoted in source JSON


## Step 2 — samplingRatio: Faster but Risky



In [3]:

import random

# Large file where first 10% has only integers, then doubles appear
rows = [{"id":i,"val":100} for i in range(9000)] + \
       [{"id":i+9000,"val":3.14} for i in range(1000)]
late_types_path = f"{DATA_DIR}/late_types.json"
with open(late_types_path,"w") as f:
    for r in rows: f.write(pyjson.dumps(r)+"\n")

# Full sampling — correct
df_full = spark.read.option("samplingRatio","1.0").json(late_types_path)
print(f"samplingRatio=1.0 → val type: {dict(df_full.dtypes)['val']}")

# 10% sampling — misses doubles in last 10%
df_sample = spark.read.option("samplingRatio","0.05").json(late_types_path)
print(f"samplingRatio=0.05 → val type: {dict(df_sample.dtypes)['val']}  ← wrong! missed doubles")

# primitivesAsString — read everything as string, cast yourself
df_str = spark.read.option("primitivesAsString","true").json(late_types_path)
print(f"primitivesAsString=true → val type: {dict(df_str.dtypes)['val']}  (safe fallback)")


samplingRatio=1.0 → val type: double
samplingRatio=0.05 → val type: double  ← wrong! missed doubles
primitivesAsString=true → val type: string  (safe fallback)


## Step 3 — Nested Object Schema Inference



In [4]:

import json as pyjson

nested_rows = [
    {"id":1,"user":{"name":"Alice","address":{"city":"NYC","zip":"10001"}},"tags":["vip","gold"],"meta":{"score":9.5}},
    {"id":2,"user":{"name":"Bob",  "address":{"city":"LA", "zip":"90001"}},"tags":["silver"],    "meta":{"score":7.0,"note":"new"}},
    {"id":3,"user":{"name":"Carol","address":{"city":"Chicago"}},          "tags":[],             "meta":None},
]
nested_path = f"{DATA_DIR}/nested.json"
with open(nested_path,"w") as f:
    for r in nested_rows: f.write(pyjson.dumps(r)+"\n")

df_nested = spark.read.json(nested_path)
print("Inferred schema for nested JSON:")
df_nested.printSchema()
df_nested.show(truncate=False)
print()
print("note: meta.note only appears in row 2 → included with null for other rows")
print("note: address.zip missing in row 3 → nullable field")


Inferred schema for nested JSON:
root
 |-- id: long (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- note: string (nullable = true)
 |    |-- score: double (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- zip: string (nullable = true)
 |    |-- name: string (nullable = true)

+---+-----------+-----------+------------------------+
|id |meta       |tags       |user                    |
+---+-----------+-----------+------------------------+
|1  |{NULL, 9.5}|[vip, gold]|{{NYC, 10001}, Alice}   |
|2  |{new, 7.0} |[silver]   |{{LA, 90001}, Bob}      |
|3  |NULL       |[]         |{{Chicago, NULL}, Carol}|
+---+-----------+-----------+------------------------+


note: meta.note only appears in row 2 → included with null for other rows
note: address.zip missing in row 3 → nullable field


## Step 4 — Explicit Schema: Production Best Practice



In [5]:

from pyspark.sql.types import *

explicit = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

import time
ndjson_path = f"{DATA_DIR}/orders.ndjson"

t0=time.time()
spark.read.json(ndjson_path).count()
t_infer = time.time()-t0

t0=time.time()
spark.read.schema(explicit).json(ndjson_path).count()
t_explicit = time.time()-t0

print(f"inferSchema  : {t_infer:.3f}s  (scans data to determine types)")
print(f"Explicit schema: {t_explicit:.3f}s  (no schema scan needed)")
print(f"Speedup      : {t_infer/t_explicit:.1f}x")
print()
print("Always use explicit schema in production.")
print("inferSchema is fine for exploration but too slow and fragile for pipelines.")


inferSchema  : 1.030s  (scans data to determine types)
Explicit schema: 0.400s  (no schema scan needed)
Speedup      : 2.6x

Always use explicit schema in production.
inferSchema is fine for exploration but too slow and fragile for pipelines.


## Summary



In [6]:

print("""
JSON schema inference summary:
  Default: reads entire file to infer types  (samplingRatio=1.0)
  samplingRatio=0.1: reads 10% — faster but may miss types in tail
  primitivesAsString=true: reads all scalars as STRING — safest fallback

Type inference rules:
  Mixed int+double → DOUBLE
  Mixed number+string → STRING
  Missing field in some rows → nullable
  null in some rows → still typed (nullable)
  Mixed objects → merged schema (union of all fields)

Production rule: ALWAYS define explicit schema for JSON pipelines.
  spark.read.schema(my_schema).json(path)

Infer once, save, reuse:
  inferred = spark.read.json(path)
  print(inferred.schema.json())  ← copy this to hardcode in your pipeline
""")



JSON schema inference summary:
  Default: reads entire file to infer types  (samplingRatio=1.0)
  samplingRatio=0.1: reads 10% — faster but may miss types in tail
  primitivesAsString=true: reads all scalars as STRING — safest fallback

Type inference rules:
  Mixed int+double → DOUBLE
  Mixed number+string → STRING
  Missing field in some rows → nullable
  null in some rows → still typed (nullable)
  Mixed objects → merged schema (union of all fields)

Production rule: ALWAYS define explicit schema for JSON pipelines.
  spark.read.schema(my_schema).json(path)

Infer once, save, reuse:
  inferred = spark.read.json(path)
  print(inferred.schema.json())  ← copy this to hardcode in your pipeline

